#6.1

In [2]:
# SciPy-based PLU (LU with permutation) + determinants + solves
import numpy as np
from scipy.linalg import lu, solve_triangular

A = np.array([[0, -4, 10],
              [-1,  6,  3],
              [ 2, -6,  7]], dtype=float)

b1 = np.array([22, 20, 11], dtype=float)
b2 = np.array([ 2, 12,  1], dtype=float)

print("Matrix A:\n", A, "\n")

# (i) PLU factorization — scipy returns P, L, U with A = P @ L @ U
P, L, U = lu(A)

print("Permutation matrix P:\n", P, "\n")
print("Lower triangular matrix L:\n", L, "\n")
print("Upper triangular matrix U:\n", U, "\n")

# Verify A = P L U
reconstructed = P @ L @ U
print("Reconstructed A (P @ L @ U):\n", reconstructed, "\n")

# (ii) Determinants
detP = np.linalg.det(P)               # should be ±1
detL = np.prod(np.diag(L))            # product of diagonal entries (1 for unit L)
detU = np.prod(np.diag(U))
detA = np.linalg.det(A)

print("det(P) =", detP)
print("det(L) =", detL)
print("det(U) =", detU)
print("det(A) =", detA)
print("Check det(A) ≈ det(P)*det(L)*det(U):", np.allclose(detA, detP*detL*detU), "\n")

# (iii) Solve A x = b using PLU:
# A x = b  =>  P L U x = b
# Multiply by P^T: L U x = P^T b
# 1) Solve L y = P^T b  (forward substitution, unit diagonal)
# 2) Solve U x = y      (back substitution)
def solve_with_plu(P, L, U, b):
    y = solve_triangular(L, P.T @ b, lower=True, unit_diagonal=True)
    x = solve_triangular(U, y, lower=False)
    return x

x1 = solve_with_plu(P, L, U, b1)
x2 = solve_with_plu(P, L, U, b2)

print("x1 solving A x1 = b1:\n", x1, "\n")
print("x2 solving A x2 = b2:\n", x2, "\n")

# Sanity checks
print("A @ x1 ≈ b1 ?", np.allclose(A @ x1, b1))
print("A @ x2 ≈ b2 ?", np.allclose(A @ x2, b2))


Matrix A:
 [[ 0. -4. 10.]
 [-1.  6.  3.]
 [ 2. -6.  7.]] 

Permutation matrix P:
 [[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]] 

Lower triangular matrix L:
 [[ 1.    0.    0.  ]
 [ 0.    1.    0.  ]
 [-0.5  -0.75  1.  ]] 

Upper triangular matrix U:
 [[ 2. -6.  7.]
 [ 0. -4. 10.]
 [ 0.  0. 14.]] 

Reconstructed A (P @ L @ U):
 [[ 0. -4. 10.]
 [-1.  6.  3.]
 [ 2. -6.  7.]] 

det(P) = 1.0
det(L) = 1.0
det(U) = -112.0
det(A) = -111.99999999999996
Check det(A) ≈ det(P)*det(L)*det(U): True 

x1 solving A x1 = b1:
 [1. 2. 3.] 

x2 solving A x2 = b2:
 [3. 2. 1.] 

A @ x1 ≈ b1 ? True
A @ x2 ≈ b2 ? True


#6.2

#6.3
### Crout's Algorithm — Explanation

Crout's algorithm computes the LU decomposition of a given invertible
square matrix $A$, meaning
$A = L U$,
where $L$ is lower triangular (with diagonal entries equal to 1)
and $U$ is upper triangular.

**Main idea:**

- We assume the matrix $A$ has an LU factorization.

- The algorithm proceeds **column by column**, iterating  
  $i = 1, 2, \dots , n$.

- For each column index $i$:

  - **Compute the upper triangular entries**
  
    All $u_{i k}$ with $k \ge i$ are computed by subtracting previously known terms:
    $$u_{i k} = a_{i k} - \sum_{j=1}^{i-1} l_{i j} u_{j k}.$$

  - **Compute the lower triangular entries**
  
    All $l_{k i}$ with $k > i$ are computed similarly:
    $$l_{k i} = \frac{ a_{k i} - \sum_{j=1}^{i-1} l_{k j} u_{j i} }{ u_{i i} }.$$

- Every new entry of $L$ or $U$ uses only values that were already computed in earlier steps.

**Summary:**  
Crout's algorithm fills the matrices $L$ and $U$ at the same time, one column at a time, always using previously computed entries.


In [1]:
# Crout LU and a Cholesky factorization (C^T C = A) implemented in pure NumPy
import numpy as np

def crout_lu(A):
    """
    Crout's LU factorization without pivoting.
    Returns L (unit diagonal), U such that A = L @ U.
    Assumes A is nonsingular and admits LU without row swaps.
    """
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n, dtype=float)
    U = np.zeros_like(A, dtype=float)

    for i in range(n):
        # Compute i-th row of U
        for k in range(i, n):
            s = 0.0
            for j in range(i):
                s += L[i, j] * U[j, k]
            U[i, k] = A[i, k] - s

        # Compute i-th column of L
        for k in range(i+1, n):
            s = 0.0
            for j in range(i):
                s += L[k, j] * U[j, i]
            if U[i, i] == 0:
                raise ZeroDivisionError("Zero pivot encountered; pivoting would be required.")
            L[k, i] = (A[k, i] - s) / U[i, i]

    return L, U

def cholesky_from_crout(A):
    """
    Cholesky factorization using a Crout-style column sweep.
    For SPD matrix A, returns C (upper triangular) such that C.T @ C = A.
    """
    A = np.array(A, dtype=float)
    n = A.shape[0]
    C = np.zeros_like(A, dtype=float)  # upper triangular

    for k in range(n):
        # Diagonal
        s = 0.0
        for j in range(k):
            s += C[j, k] * C[j, k]
        val = A[k, k] - s
        if val <= 0:
            raise ValueError("Matrix is not symmetric positive definite.")
        C[k, k] = np.sqrt(val)

        # Off-diagonals in column k (positions (k, i) with i > k)
        for i in range(k+1, n):
            s = 0.0
            for j in range(k):
                s += C[j, k] * C[j, i]
            C[k, i] = (A[k, i] - s) / C[k, k]

    return C

# Demo
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)
M = rng.normal(size=(5,5))
A_spd = M.T @ M + 5*np.eye(5)        # SPD test matrix

L, U = crout_lu(A_spd)
C = cholesky_from_crout(A_spd)

print("A (SPD):\n", A_spd, "\n")
print("Crout LU check ||A - L U||:", np.linalg.norm(A_spd - L @ U))
print("Cholesky check ||A - C^T C||:", np.linalg.norm(A_spd - C.T @ C), "\n")
print("L (Crout, unit diag):\n", L, "\n")
print("U (Crout):\n", U, "\n")
print("C (upper; C^T C = A):\n", C)


A (SPD):
 [[ 8.3344  0.0096  1.2965  0.5735  1.0495]
 [ 0.0096  7.3025 -1.9556  0.3256  2.5569]
 [ 1.2965 -1.9556  8.0816 -0.2457 -1.6393]
 [ 0.5735  0.3256 -0.2457  7.9512 -1.2714]
 [ 1.0495  2.5569 -1.6393 -1.2714  9.9387]] 

Crout LU check ||A - L U||: 4.47545209131181e-16
Cholesky check ||A - C^T C||: 1.778090716669452e-15 

L (Crout, unit diag):
 [[ 1.      0.      0.      0.      0.    ]
 [ 0.0012  1.      0.      0.      0.    ]
 [ 0.1556 -0.268   1.      0.      0.    ]
 [ 0.0688  0.0445 -0.0337  1.      0.    ]
 [ 0.1259  0.35   -0.1519 -0.1895  1.    ]] 

U (Crout):
 [[ 8.3344  0.0096  1.2965  0.5735  1.0495]
 [ 0.      7.3025 -1.9571  0.3249  2.5557]
 [ 0.      0.      7.3554 -0.2478 -1.1176]
 [ 0.      0.      0.      7.8889 -1.495 ]
 [ 0.      0.      0.      0.      8.459 ]] 

C (upper; C^T C = A):
 [[ 2.8869  0.0033  0.4491  0.1987  0.3635]
 [ 0.      2.7023 -0.7242  0.1202  0.9457]
 [ 0.      0.      2.7121 -0.0914 -0.4121]
 [ 0.      0.      0.      2.8087 -0.5323]
 [ 